# Task 2 — Submission Generation

**Purpose:** Generate Kaggle submission CSV from trained model.

**Usage:**
1. Train a model using `Train_notebook.ipynb`
2. Run this notebook to generate predictions
3. Submission saved to `generated_data/current_results/task2_submission.csv`

**Runtime:** < 1 minute


In [4]:
import torch
import torch.nn as nn
import pandas as pd
import os
from torch.utils.data import DataLoader
import importlib

# Reload modules to ensure latest changes are loaded
import models
import datasets
importlib.reload(models)
importlib.reload(datasets)

from models import create_efficientnet_b3
from datasets import KaggleTestDataset300, INV_LABELS_DICT

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# Create output directory
os.makedirs("generated_data/current_results", exist_ok=True)


Device: cuda


## Configuration


In [5]:
# Model path - using fine-tuned model from fine_tune_results
model_path = "generated_data/fine_tune_results/fine_tuned_model.pth"

if not os.path.exists(model_path):
    raise FileNotFoundError(
        f"Model not found at {model_path}. Please ensure the fine-tuned model exists."
    )

print(f"Using fine-tuned model from fine_tune_results")

batch_size = 128
num_workers = 4
persistent_workers = True
prefetch_factor = 2
pin_memory = True

print(f"Model path: {model_path}")


Using fine-tuned model from fine_tune_results
Model path: generated_data/fine_tune_results/fine_tuned_model.pth


## Load Model and Dataset


In [6]:
# Load test dataset
kaggle_test_dataset = KaggleTestDataset300()

kaggle_test_loader = DataLoader(
    kaggle_test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory,
    persistent_workers=persistent_workers,
    prefetch_factor=prefetch_factor
)

print(f"Test samples: {len(kaggle_test_dataset)}")

# Load model
model = create_efficientnet_b3(num_classes=10).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print("Model loaded successfully!")


Test samples: 2000
Model loaded successfully!


## Generate Predictions


In [7]:
all_filenames = []
all_pred_indices = []

print("Generating predictions...")
with torch.no_grad():
    for x, fnames in kaggle_test_loader:
        x = x.to(device)
        logits = model(x)
        _, cls = logits.max(1)
        all_filenames.extend(list(fnames))
        all_pred_indices.extend(cls.cpu().numpy().tolist())

# Map indices to label strings
all_pred_labels = [INV_LABELS_DICT[idx] for idx in all_pred_indices]

print(f"Generated {len(all_pred_labels)} predictions")


Generating predictions...
Generated 2000 predictions


## Save Submission File


In [8]:
submission_df = pd.DataFrame({
    "id": all_filenames,
    "label": all_pred_labels
})

submission_df = submission_df.sort_values("id")

output_path = "generated_data/current_results/task2_submission.csv"
submission_df.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"Shape: {submission_df.shape}")
print("\nFirst few predictions:")
print(submission_df.head(10))


Submission saved to: generated_data/current_results/task2_submission.csv
Shape: (2000, 2)

First few predictions:
          id       label
0  00000.png         cat
1  00001.png       truck
2  00002.png       horse
3  00003.png    airplane
4  00004.png  automobile
5  00005.png        bird
6  00006.png       horse
7  00007.png        deer
8  00008.png  automobile
9  00009.png  automobile
